In [7]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250, add_mask_inside_swot, build_swath_polygon

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

/Users/mdemol/code/pynsitu/pynsitu/__init__.py:45: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  hour = Timedelta("1H")


In [ ]:
def compute_box_thetas(ds):
    """Compute the local angles theta between e_lon and x and between e_lat and x for the box of one colocalisation
    https://github.com/rasterio/affine

    Parameters
    ----------
    ds : xr.DataArray
        colocalisations dataset
    Return
    ------
    box_theta_longitude, box_theta_latitude : np.array, np.array


    """

    dlon_dx, dlon_dy = ds.box_lon.differentiate("box_x"), ds.box_lon.differentiate(
        "box_y"
    )
    dlat_dx, dlat_dy = ds.box_lat.differentiate("box_x"), ds.box_lat.differentiate(
        "box_y"
    )

    return np.arctan2(-dlat_dx, dlat_dy), np.arctan2(dlon_dx, -dlon_dy)


def compute_drifters_thetas_core(lonc, latc, phi, lonxy, latxy, x, y, dx=100, dy=100):
    """Compute the local angles theta between e_lon and x and between e_lat and x for drifters locations of one colocalisation
    https://github.com/rasterio/affine

    Parameters
    ----------
    lonc, latc, phi: float
    central position and orientation of the box
    lonxy, latxy : drifters position in longitude, latitude
    x, y: np.array position of drifters in local coordinates (with origin at (lonc, latc) and x-axis aligned
    with the satellite track (lonc, latc) - (lon1, lat1) direction)
    dx, dy : differential, default is 100m

    Return
    ------
    theta_longitude, theta_latitude : np.array, np.array


    """
    proj = get_proj(lonc, latc)
    # assert False, help(proj.transform)
    xc, yc = proj.transform(lonc, latc)
    # apply inverse affine transformation
    a_fwrd = Affine.translation(-xc, -yc) * Affine.rotation(-phi, pivot=(xc, yc))
    a_back = ~a_fwrd
    # compute coordinates of x,y in the lon, lat orientated grid
    x_invx, y_invx = a_back * (x + dx, y)
    lonx, latx = proj.transform(
        x_invx,
        y_invx,
        direction=pyproj.enums.TransformDirection.INVERSE,
    )
    dlon_dx, dlat_dx = (lonx - lonxy) / dx, (latx - latxy) / dx

    x_invy, y_invy = a_back * (x, y + dy)
    lony, laty = proj.transform(
        x_invy,
        y_invy,
        direction=pyproj.enums.TransformDirection.INVERSE,
    )
    dlon_dy, dlat_dy = (lony - lonxy) / dy, (laty - latxy) / dy

    theta_lon = np.arctan2(-dlat_dx, dlat_dy)  # angles between e_lon and x
    theta_lat = np.arctan2(dlon_dx, -dlon_dy)  # angles between e_lat and x
    return theta_lon, theta_lat


In [3]:
dfs = browse_swot_250().reset_index()
drifters_sources = 'all_med_variational_10min_v0.nc'
df = pd.read_csv(os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv')))

/var/folders/fn/z858c2qj1lz65xr0z5mdvbf40000gp/T/ipykernel_12121/2062596265.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv')))


In [17]:
# get grid orientation and metrics
def add_grid_metrics(ds):
    """ add grid spatial metrics """

    geod = Geod(ellps="WGS84")

    lon, lat = ds.longitude, ds.latitude
    dims = lon.dims
    
    # d/dx where x is cross-track
    az12, az21, dx = geod.inv(
        lon, lat, lon.shift(num_pixels=-1), lat.shift(num_pixels=-1),
    )
    
    ds = ds.assign_coords(dx=(dims, dx), phi=(dims, az12*np.pi/180))
    
    ds["dx"] = (
        ds["dx"]
        .ffill("num_pixels")
        .where(ds["duacs_editing_flag"]<5)
    )

    # phi is cross-track direction from north
    ds["phi"] = (
        ds["phi"]
        .ffill("num_pixels")
        .where(ds["duacs_editing_flag"]<5)
    )

    # d/dy where y is along-track
    
    az12, az21, dy = geod.inv(
        lon, lat, lon.shift(num_lines=-1), lat.shift(num_lines=-1),
    )
    
    ds = ds.assign_coords(dy=(dims, dy))
    
    ds["dy"] = (
        ds["dy"]
        .ffill("num_lines")
        .where(ds["duacs_editing_flag"]<5)
    )
    
    return ds

In [18]:
cycle = 500
swath = 3

dss = xr.open_dataset(dfs.where((dfs.pass_number==swath)&(dfs.cycle_number==cycle)).dropna().file.values[0])
dss = add_grid_metrics(dss)

/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


In [21]:
dss.phi.mean()*180/np.pi

<xarray.DataArray 'phi' ()> Size: 8B
array(103.20892515)

In [22]:
dss.phi2.mean()*180/np.pi

<xarray.DataArray 'phi2' ()> Size: 8B
array(-76.78923859)

In [ ]:
def vevn2vxvy(theta_lon, theta_lat, ve, vn):
    """Compute velocities projected on the local box grid (ve, vn -> vx-along satellite track, vy-normal to satellite track)
    https://github.com/rasterio/affine

    Parameters
    ----------
    lonc, latc, phi: float  central position and orientation of the box
    ve,vn: np.array velocities on lon, lat grid
    theta_lon, theta lat

    Return
    ------
    local vx, local vy : np.array, np.array

    """
    return ve * np.cos(theta_lon) + vn * np.cos(theta_lat), ve * np.sin(theta_lon) + vn * np.sin(theta_lat)